In [9]:
import torch 
import pandas as pd
from sklearn.model_selection import train_test_split

In [11]:
df = pd.read_csv('data/hospital_ml_ready.csv')
df

,datetime,energy_kwh,hour,day_of_week,is_weekend,prev_hour_energy,rolling_3h_avg,rolling_6h_avg,target_next_hour
0,2023-01-01 05:00:00,424.98,5,6,1,424.98,442.550000,442.305000,749.38
1,2023-01-01 06:00:00,749.38,6,6,1,424.98,533.113333,492.718333,725.02
2,2023-01-01 07:00:00,725.02,7,6,1,749.38,633.126667,542.246667,687.92
3,2023-01-01 08:00:00,687.92,8,6,1,725.02,720.773333,581.661667,718.28
4,2023-01-01 09:00:00,718.28,9,6,1,687.92,710.406667,621.760000,688.10
...,...,...,...,...,...,...,...,...,...
8749,2023-12-31 18:00:00,520.32,18,6,1,690.73,643.873333,678.493333,578.08
8750,2023-12-31 19:00:00,578.08,19,6,1,520.32,596.376667,659.391667,525.08
8751,2023-12-31 20:00:00,525.08,20,6,1,578.08,541.160000,625.500000,548.82
8752,2023-12-31 21:00:00,548.82,21,6,1,525.08,550.660000,597.266667,436.90


In [12]:
feature_cols = ["hour", "day_of_week", "is_weekend",
                 "prev_hour_energy", "rolling_3h_avg", "rolling_6h_avg"]

In [13]:
X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
y = torch.tensor(df["target_next_hour"].values, dtype=torch.float32)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [16]:
mean = X_train.mean(dim=0)
std = X_train.std(dim=0)
X_train_norm = (X_train - mean)/ std
X_test_norm = (X_test - mean) / std

In [17]:
weight = torch.zeros(len(feature_cols), requires_grad=True)
bias = torch.zeros(1, requires_grad=True)

In [18]:
lr = 0.01

In [20]:
for epoch in range(300):
    y_pred = X_train_norm @ weight + bias
    loss = ((y_pred - y_train) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        weight -= lr * weight.grad
        bias -= lr * bias.grad

    weight.grad.zero_()
    bias.grad.zero_()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}: train loss={loss.item():.2f}")

    with torch.no_grad():
        test_pred = X_test_norm @ weight + bias
        test_loss = ((test_pred - y_test) ** 2).mean()

print("Test loss:", test_loss.item())
print("Learned weights:", weight.detach())

Epoch 0: train loss=352586.25
Epoch 50: train loss=52322.72
Epoch 100: train loss=13328.95
Epoch 150: train loss=8054.87
Epoch 200: train loss=7284.68
Epoch 250: train loss=7121.81
Test loss: 6883.87890625
Learned weights: tensor([-42.1074,  -1.0753,  -1.5954,  36.6427,  76.2473,  10.6067])


In [23]:
with torch.no_grad():
    latest_features = X_test_norm[-1].unsqueeze(0)
    predicted_next_hour = latest_features @ weight + bias
    actual_next_hour = y_test[-1]

print("Predicted next-hour energy (kWh):", predicted_next_hour.item())
print("Actual next-hour energy (kWh):", actual_next_hour.item())

Predicted next-hour energy (kWh): 457.68768310546875
Actual next-hour energy (kWh): 475.8500061035156
